# A custom encrypted model, all from the Python binding

`perseus` runs neural networks under CKKS fully homomorphic encryption on GPU
(FIDESlib + OpenFHE). This notebook defines a **custom model in pure Python** with
torch-style `Enc*` modules, runs it **encrypted end-to-end**, then closes the
production loop: **capture** the operation graph, compute a **custom bootstrap plan**
with the min-cut planner, and runs it.

In [ ]:
import os

import numpy as np

from perseus import _core
from perseus.nn import EncGELU, EncLinear, EncSequential

opts = _core.InferenceOptions()
opts.ckks = _core.CKKSOptions.from_env()
opts.mode = _core.InferenceMode.Sync
inf = _core.make_gpt2_inference(opts)   # GPT-2 consumes eccessive memory if the model is small, an adaptive key collector will be implemented.
print(f"slots={inf.slots}  logN={inf.logN}  level_limit={inf.fhe.level_limit()}")

[ckks_env] recognized environment knobs (effective values):
  LOGN                  = 16  [env]
  CKKS_DEPTH            = 11  [def]
  BTP_DEPTH_OVERHEAD    = 16  [def]
  FIRST_MOD_BITS        = 60  [def]
  BTP_SCALE_BITS        = 53  [def]
  SCALE_BITS            = 58  [def]
  CORRECTION_FACTOR     = 0  [def]
  NUM_LARGE_DIGITS      = 7  [def]
  AUTO_BTS_LEVEL        = 24  [env]
  H_WEIGHT              = 192  [def]
  BTS_ITERATIONS        = 1  [env]
  BTS_PRECISION         = 12  [def]
  CKKS_COMPLEX          = 1  [env]
  SPARSE_BTS_SLOTS      = 0  [def]
  LEVEL_BUDGET          = 4:3  [def]
  limb_chain            = (uniform 53)  [def]


[ckks_params] ===== CKKS crypto context =====
  logN=16  ringDim=65536  slots=32768
  depth=11  btp_overhead=16  mult_depth=27  limbs=28
  first_mod=60  scale(active)=53  (btp_scale=53 scale_bits=58)  logQ~=1491
  bootstrap=on  correction_factor=0  bts_iters=1  bts_prec=12  auto_bts_level=24
  secret=SPARSE_ENCAPSULATED h=192 (OpenFHE accounts UNIFORM_TERNARY)  dnum=7  complex_payload=1
  security: HEStd_128_classic enforced & OK (logQ 1491 <= uniform-ternary budget 1761)
[ckks_params] ===============================


GPU 0: NVIDIA A100-SXM-64GB
 SMs: 124, SharedMem: 164 KB, Blocks/SM: 32, Threads/SM: 2048, L2 size: 32 MB, Bus: 4096-bit


[rot_band] band=11 full_keep=76 (precom entries=1)


[rot_band] model keys: banded=87 full=46


Adding bootstrap precomputation to GPU for 32768 slots.
Plaintexts loaded: 313 ~ 3623MB
Rotation keys loaded: 163 ~ 36512MB
slots=32768  logN=16  level_limit=24


## Define the model

An encrypted expand/contract MLP (1024 → 4096 → GELU → 1024 padded; 768/3072 real), like a transformer MLP. The GELU is a
*calibration-carrying* module: its polynomial approximation comes from a fitted section
of `configs.json`. 

Weights are plain numpy matrices `(d_in, d_out)`, encrypted at bind
time.

In [ ]:
configs = _core.load_configs(os.environ["CONFIGS_PATH"]) # must be generated with the setup_artifacts notebook.

d_pad, d_real = 1024, 768
d_exp, e_real = 4096, 3072
rng = np.random.default_rng(0)


def make_weight(d_in, d_out, in_real, out_real, scale=0.5):
    w = rng.standard_normal((d_in, d_out)) * scale / np.sqrt(in_real)
    w[in_real:, :] = 0.0
    w[:, out_real:] = 0.0
    return w


W1 = make_weight(d_pad, d_exp, d_real, e_real)
W2 = make_weight(d_exp, d_pad, e_real, d_real)

# torch style model definition
model = EncSequential(
    EncLinear("fc1", d_pad, d_exp, weight=W1.tolist()),
    EncGELU("act"),
    EncLinear("fc2", d_exp, d_pad, weight=W2.tolist()),
).bind(inf)
inf.set_gelu_cfg("act", configs.softgelu["transformer.h.0.mlp.act"])
print("model bound: fc1(1024->4096) -> gelu -> fc2(4096->1024), weights encrypted on device")

model bound: fc1(1024->4096) -> gelu -> fc2(4096->1024), weights encrypted on device


## Encrypted forward vs plaintext reference

Encrypt one input vector, run the model **eagerly** (the runtime fires bootstraps
automatically whenever a ciphertext runs out of levels), and compare against the exact
plaintext computation.

In [3]:
x_real = rng.standard_normal(d_real) * 0.3


def gelu_exact(v):
    return 0.5 * v * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (v + 0.044715 * v**3)))


x_pad = np.zeros(d_pad)
x_pad[:d_real] = x_real
ref = (gelu_exact(x_pad @ W1) @ W2)[:d_real]   # (d_in, d_out) weights: y = x @ W

x = _core.encode_token_input(inf, x_real.tolist())
y = model(x)
out = np.array(_core.decode_token_output(inf, y))[:d_real]

rel = np.linalg.norm(out - ref) / np.linalg.norm(ref)
print(f"eager encrypted forward: rel_err={rel:.3e}  (in L{x.level} -> out L{y.level})")
assert rel < 0.05

[bts-level] btp_overhead=16 bts_iterations=1 params_formula=16 probed=16 -> encoding weights at level 16


eager encrypted forward: rel_err=7.968e-03  (in L16 -> out L17)


## Capture the operation graph

Every CKKS primitive the model executes is recorded: op type, operands, levels, and
observed magnitudes. The captured graph is the planner's single source of truth.

In [4]:
graph_dir = ".cache/custom_seq/block_0"
os.makedirs(graph_dir, exist_ok=True)

_core.reset_graph_runtime(inf)
inf.enable_graph_capture()
model(_core.encode_token_input(inf, x_real.tolist()))
inf.export_graph_json(f"{graph_dir}/graph.json")
inf.disable_graph_capture()

import json

nodes = json.load(open(f"{graph_dir}/graph.json"))["nodes"]
eager_bts = sum(n["op_type"] == "auto_bootstrap" for n in nodes)
print(f"captured {len(nodes)} nodes, {eager_bts} eager auto-bootstraps")

captured 810 nodes, 0 eager auto-bootstraps


## Plan: a custom bootstrap schedule

`perseus.plan` runs a range-aware min-cut over the captured graph: it erases the
eager greedy bootstraps and re-places them on magnitude-safe, level-optimal variables.
The output is a placement plan an artifact bound to this exact model.

In [ ]:
from perseus.plan import PlanOptions, plan_graph_dir

plan_dir = "bootstrap_placements/custom_seq"
summaries = plan_graph_dir(".cache/custom_seq", plan_dir, PlanOptions())

planned_bts = sum(s["num_placements"] for s in summaries)
print(f"planned placements: {planned_bts} (eager fired {eager_bts} auto-bootstraps)")

out=bootstrap_placements/custom_seq max_abs=50.0 max_level=24 cache_read_level=17
[block_0] -> bootstrap_placements/custom_seq/block_0_placement.json (entry: capture-derived, bts_level=16)
[min_cut] erase-autobts: dropped 0 reactive auto_bootstrap (min-cut re-places, incl. out-of-band w/ in-band ancestor); kept 0 (unsafe |abs|>50.0 + precision-keep []) as deliberate
[min_cut] per-op caps active on 2 nodes (level_hint)
[min_cut] Running layer-by-layer min-cut (L=8)...
[min_cut] weight levels: (none — weight steps not found in graph)
[min_cut] mask levels: gelu.half=[23]
  bootstraps=3 placements=0
planned placements: 0 (eager fired 0 auto-bootstraps)


## Planned deterministic rerun

Load the plan and run again. Planned mode is **strict**: every bootstrap fires exactly
where the plan says, and any divergence between model and plan raises instead of
silently degrading.

In [6]:
_core.reset_graph_runtime(inf)
inf.load_bootstrap_plan_json(f"{plan_dir}/block_0_placement.json")   # hint-only plans (0 min-cut placements) return False but still bind the hint decisions

y_planned = model(_core.encode_token_input(inf, x_real.tolist()))
inf.clear_bootstrap_plan()

out_planned = np.array(_core.decode_token_output(inf, y_planned))[:d_real]
rel_planned = np.linalg.norm(out_planned - ref) / np.linalg.norm(ref)
print(f"planned forward: rel_err={rel_planned:.3e}  placements {eager_bts} -> {planned_bts}")
assert rel_planned < 0.05

planned forward: rel_err=8.261e-03  placements 0 -> 0


## Performance

Per-forward wall time (encrypt → forward → decrypt), eager vs planned, on one A100.
This chain is shallow enough that eager never runs out of levels, so here the plan buys
determinism and the timing shows the raw cost of its placements.
At depth the ranking flips: bootstraps dominate deep models latency and the planner places them optimally (encrypted
GPT-2: eager 129.5 s/token vs planned 50.6, see the GPT-2 inference notebook).

In [7]:
import time


def one_forward():
    _core.reset_graph_runtime(inf)   # restart var naming so the strict plan re-binds
    y = model(_core.encode_token_input(inf, x_real.tolist()))
    return _core.decode_token_output(inf, y)


def timeit(n=5):
    one_forward()   # warm
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        one_forward()
        ts.append(time.perf_counter() - t0)
    return np.mean(ts) * 1e3, np.std(ts) * 1e3


eager_ms = timeit()
inf.load_bootstrap_plan_json(f"{plan_dir}/block_0_placement.json")   # hint-only plans (0 min-cut placements) return False but still bind the hint decisions
planned_ms = timeit()
inf.clear_bootstrap_plan()

print(f"eager   forward: {eager_ms[0]:6.0f} ± {eager_ms[1]:.0f} ms   ({eager_bts} eager bootstraps)")
print(f"planned forward: {planned_ms[0]:6.0f} ± {planned_ms[1]:.0f} ms   ({planned_bts} planned placements)")
print(f"device memory free: {_core.device_free_gb():.1f} GB")

eager   forward:    619 ± 1 ms   (0 eager bootstraps)
planned forward:    618 ± 1 ms   (0 planned placements)
device memory free: 33.3 GB


## What this showed

- A model **defined only in Python** (torch-style modules, numpy weights) ran fully
  encrypted on the CUDA FHE runtime.
- The **capture → plan → planned-strict** production loop closed entirely from Python:
  the same planner artifacts that drive the GPT-2 baselines now bind custom models.
- Encrypted-forward latency measured directly, eager vs planned, on one A100.